# Exploratory Data Analysis - M-AxPPA Trade-off Dataset

This notebook explores the synthetic dataset used in the Approximate Computing Trade-off Explorer project. The goal is to show the first analytical steps behind the Power BI and Streamlit dashboards.

## Context

The dataset is synthetic and based on the public structure of the M-AxPPA paper. It does not represent real hardware measurements. It is used to demonstrate data preparation, exploratory analysis, SQL/BI integration, and decision-oriented trade-off analysis.

In [ ]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASET_PATH = ROOT / "data" / "processed" / "tradeoff_dataset.csv"

df = pd.read_csv(DATASET_PATH)
df.head()

## Dataset Overview

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df[[
    "mred",
    "energy_saving_pct",
    "area_saving_pct",
    "balanced_score",
]].describe().T

## Architecture Mix

The next cells show how many architectures exist by family and variant. This helps verify whether the dataset has enough variation for comparative analysis.

In [ ]:
df.groupby(["family", "variant"]).size().reset_index(name="architectures")

## Trade-off Summary By Variant

In [ ]:
variant_summary = (
    df.groupby("variant")
    .agg(
        architectures=("architecture_id", "count"),
        median_mred=("mred", "median"),
        max_energy_saving=("energy_saving_pct", "max"),
        max_area_saving=("area_saving_pct", "max"),
        mean_balanced_score=("balanced_score", "mean"),
    )
    .sort_values("mean_balanced_score", ascending=False)
)

variant_summary

## Controlled Error Analysis

A common decision rule is to filter candidates by acceptable error first. The README uses `MRED <= 0.10` as a representative threshold.

In [ ]:
controlled = df[df["mred"] <= 0.10].copy()
controlled.shape

In [ ]:
controlled.sort_values("energy_saving_pct", ascending=False)[[
    "family",
    "variant",
    "m_bits",
    "l_bits",
    "k_bits",
    "mred",
    "energy_saving_pct",
    "area_saving_pct",
    "balanced_score",
]].head(10)

In [ ]:
controlled.sort_values("area_saving_pct", ascending=False)[[
    "family",
    "variant",
    "m_bits",
    "l_bits",
    "k_bits",
    "mred",
    "energy_saving_pct",
    "area_saving_pct",
    "balanced_score",
]].head(10)

## Pareto Candidates

Pareto candidates help identify architectures that are not dominated in error-energy or error-area trade-offs.

In [ ]:
pareto = df[
    (df["pareto_optimal_energy_error"] == 1)
    | (df["pareto_optimal_area_error"] == 1)
].copy()

pareto[[
    "family",
    "variant",
    "mred",
    "energy_saving_pct",
    "area_saving_pct",
    "pareto_optimal_energy_error",
    "pareto_optimal_area_error",
]].sort_values(["mred", "energy_saving_pct"], ascending=[True, False]).head(20)

## Initial Takeaways

- The best candidate changes when the ranking criterion changes.
- Filtering by error before ranking makes the decision more realistic.
- Pareto flags are useful for narrowing the candidate set.
- The same dataset can support SQL analysis, Power BI reporting, and Streamlit exploration.